# Data Processing Code

- Author: RUEI-HSIANG, CHANG／張睿庠（B12303030；AI協助）
- Date: 2026-11-08
- Purpose: 從全國1583筆登記母表，依鍵覆核兩市210人新版，再做本輪合理範圍的G5／G6補查，產生全國合併／首長／議員三池。保留所有登記者稽核，不把未查的G6填0。
- Inputs: 精確檔名與SHA在下方；[原全國文件](../processed/clustering_20261108_documentation.ipynb)、[兩市暫納入文件](../processed/taipei_kaohsiung_yu_provisional_20261108_documentation.ipynb)、[本輪raw文件](../raw/taiwan_completion_20261108_documentation.ipynb)。
- Rules: 楊喬茵修訂G5；G6只計配偶／三親等內民選或政治任命公職，一般軍文職排除；完成搜尋未發現符合證據可0。資格採既有10/16快照，除學生明確暫納入游智彬外，其他9位資格不符議員排除。年齡／學歷缺值不刪除。
- Bounded search: 49位G6未明首長與14位先前零完整議員縣市的人選各兩輪搜尋，加4項全文中的親屬關係；636位G5未明全數掃描已存145份歷史檔，並覆核3個生日＋學校／參選經歷連結。不是已查完所有全國G6。
- Output / verification: 新檔皆在data/processed，序列化後重新開啟核對每格、鍵、三池分割、22縣市範圍與缺值；舊檔不覆寫。正式分群另在finding；此處不估模型。

In [1]:
from pathlib import Path
import hashlib,json,csv,re
import pandas as pd
from bs4 import BeautifulSoup
from IPython.display import display,Markdown
ROOT=next((p for p in (Path.cwd(),*Path.cwd().parents) if (p/'AGENTS.md').is_file() and (p/'templates').is_dir()),None)
assert ROOT is not None
sha=lambda p:hashlib.sha256(p.read_bytes()).hexdigest()
KEY='candidate_key';G5='G5_first_time_any_elected_office';G6='G6_political_family'

In [2]:
PINS={'data/processed/clustering_all_20261108.csv': '2a70db4a868a80b35934d0437196cc8f189d27c6625e7a67b0bb6e753c0305bc', 'data/processed/taipei_kaohsiung_yu_provisional_all_20261108.csv': '35e3b85e407d4eb91e5c96043d06a407d21e070a5920032b26f74e916f883cbe', 'data/raw/cec_all_offices_manifest.json': '9cc85f7f913f5de717ba92ec7698c73cedb29df506fba59a69933d1ac3517231', 'data/raw/taipei_kaohsiung_completion_20261108/cec_eligibility.json': '342c46119447371eb571b6cc22ea59b5b28453f75a64bd556d95b70f263ef920'}
RAW_PINS={'data/raw/taiwan_completion_20261108/G5_identity_adjudications.json': 'a6242bb8b1c5880bc40e9e410b02cba46694db2a281fabbc6a4850ce3167c57e', 'data/raw/taiwan_completion_20261108/G6_adjudications.json': 'be2b800105053fd31e7022c9591c5c5b1057c2b6e10336b977f31962e4155475', 'data/raw/taiwan_completion_20261108/article_01.txt': '4769a3f9ba376fd0938be45620dc699621b4e46c8e84e50b4e7aa1a39d17eb6b', 'data/raw/taiwan_completion_20261108/article_02.txt': '79e284265fe56319d0b8f32b8337f5a30f92987ea85f815efb51a3b1e8814116', 'data/raw/taiwan_completion_20261108/article_03.txt': '435bc89dc87b8cab6c56cf9cb5eb88479ab0db8488e610ea7d8eba873945019e', 'data/raw/taiwan_completion_20261108/article_04.txt': '9c3c917cd3a01d5ccb873fbb5d25a40afe11d987cbae98dcf6716a8b92046b0a', 'data/raw/taiwan_completion_20261108/article_05.txt': 'ed41d9b7c672686656a4d5a973a84b4b10540051bee67593faa3154621e34054', 'data/raw/taiwan_completion_20261108/article_06.txt': '495a2202c2484be3bf776a1264497afbb0163c67b6dd8a0c398c9b369123913d', 'data/raw/taiwan_completion_20261108/article_07.txt': '4a7b0fa47cab97b0a99ff1612d66d261f01826f8585dea305518b1661c7cfcf6', 'data/raw/taiwan_completion_20261108/article_08.txt': '96ed7780c12a4cca24c9c9161f70dbb4db9f76beccc76884b1c4d98d47ae5a89', 'data/raw/taiwan_completion_20261108/article_09.txt': '80db0830c3eb5ea5a2c99640cf8bb74eaa467107e43f3a271ddf4e117716f3d2', 'data/raw/taiwan_completion_20261108/article_10.txt': '344a9ca67d02f6220499ae7e165379f532fe46689bd25618325803479ddba3e2', 'data/raw/taiwan_completion_20261108/article_11.txt': 'f83fe2fc108763caf9d97211c1bcb3fa0053db052ad6e73e3c9185aa401a1b33', 'data/raw/taiwan_completion_20261108/article_12.txt': '9f46a0a62d67e44075f64cf130c4c91e7aef4f2844e956f05c99850dc840c434', 'data/raw/taiwan_completion_20261108/article_13.txt': '0a6b30eab2b2305f0f0c7051573ff6c6d0b3bdb53f60fc0c8950093812315601', 'data/raw/taiwan_completion_20261108/article_14.txt': '38895d7dd59665e9ee15de32519d07098891d0b9659ba539429d57cf8036a9d2', 'data/raw/taiwan_completion_20261108/article_15.txt': '4c82dc8980570a983490ff5d4e79c9dec93e2f475a6713c5de52f218d9b0617e', 'data/raw/taiwan_completion_20261108/article_16.txt': '884f1816558c33a685e781eb6e91d7bd1254089f5a6662368daccd526e1f3cad', 'data/raw/taiwan_completion_20261108/article_17.txt': '11316ff7d246df1f2e9da2ac8c94ebbf38a3228182d3f3647c1bb429325ea6fa', 'data/raw/taiwan_completion_20261108/article_18.txt': '292dfd3ff706b2371a9f7080cd6d05a5d02e9dda1328f7924a9d3c167840587c', 'data/raw/taiwan_completion_20261108/article_19.txt': 'dfb14725e86f9ffb844956a3e96dd0bbf6e4f79242f3bb6c4dbff2daf6523999', 'data/raw/taiwan_completion_20261108/article_20.txt': '2684dd18cba809a4f9e9f523f557c10aacc54b9973299297d5d550d2c68aa198', 'data/raw/taiwan_completion_20261108/article_21.txt': '51747524ec6472e10dfb424e6abab96a48d0b019ced4237671e9d8608937ec60', 'data/raw/taiwan_completion_20261108/article_22.txt': 'b03f11819cf64b81c49d4b002bbf7cd785853b6ad64d91c9eea61a73b4d9789c', 'data/raw/taiwan_completion_20261108/article_23.txt': 'c58116e80f98011f621096156b8a7fd4100e8d8d1e64715ea201f85b3e7b25e3', 'data/raw/taiwan_completion_20261108/article_24.txt': '83b6e87076f6937fd1f2cc6378b907db7db27087725965379bdc6b1462744c23', 'data/raw/taiwan_completion_20261108/article_25.txt': 'b4b6dab43acd99b67e86fec4671ef7700f43b1290699992acf7887585f7ac493', 'data/raw/taiwan_completion_20261108/article_26.txt': 'b46c838668c435954af6f15788670cdd1ea56ece87da29632c0d38fe4aac5451', 'data/raw/taiwan_completion_20261108/article_27.txt': 'd9e7b89c3836c935260e20e989af4246807ff92d105e5dd2ea57b6d1cf73d65f', 'data/raw/taiwan_completion_20261108/article_28.txt': 'e9f421c0966a6ab79e6a99271a9143f40a83754d5e85bfd2cc1bb31f0971148d', 'data/raw/taiwan_completion_20261108/article_29.txt': 'c644bc16b395ebc2492b36db4dcc594c6f7cdbeda25341e98e0f18bee57a69a2', 'data/raw/taiwan_completion_20261108/article_30.txt': 'e78df75f63d9d9d4edea1037209fd7c7eab9df5d8b4551f2f924597cc57ccdd0', 'data/raw/taiwan_completion_20261108/article_31.txt': '695829bd71bb8ed8360895f42e1832035534c26db1845151324b887b2f5e532e', 'data/raw/taiwan_completion_20261108/article_32.txt': 'f7edfb276341b2b8e8208c5af6aa250173599002a651ff3ad77fd15cc8a92bb4', 'data/raw/taiwan_completion_20261108/article_33.txt': '60dc553863ae4d608fbf6e401852f0b732b5d44a9f2bfc3a9f360b6ab8dfd56b', 'data/raw/taiwan_completion_20261108/article_34.txt': '6c505458522f889071890b000b5e44606f41a7ae5fb549581269283d03a24203', 'data/raw/taiwan_completion_20261108/article_35.txt': '6a1b0b233380b318241ec0937e5c132ff4fdd7d88ad4b9fd277fd284232d6028', 'data/raw/taiwan_completion_20261108/article_36.txt': 'a1b0672e4f4f0d0819be761235c0dbe1901b0dd1ffe273f59e15af24a2dd39d6', 'data/raw/taiwan_completion_20261108/article_37.txt': 'af0bfd7335f93c2cbf15523600bdaf20dea711fd70e3c59a4f7a7bac53af367f', 'data/raw/taiwan_completion_20261108/article_38.txt': '260c4ad837879e79ddc3204b0c95588d7e77202a0dcea8db91e8672f6d87273a', 'data/raw/taiwan_completion_20261108/article_39.txt': '2868c20577758a67aad841736bf9085f8843160b0513c69c997ed2ba0dda8a4b', 'data/raw/taiwan_completion_20261108/article_40.txt': '53bf481599a22b347f736542be1674057df73403c7ffbdf540e76c562ab65b3d', 'data/raw/taiwan_completion_20261108/article_41.txt': 'e64f74d1dded2a152c9e32f709980832c4f6d8d1ca0dda578fa7a7859196a07e', 'data/raw/taiwan_completion_20261108/article_manifest.json': '896ba18f861bd77718a6eeaa5562fd595b688747ab8628c227c297244527fb5f', 'data/raw/taiwan_completion_20261108/article_open_00.txt': '906fad1ffaf0e6515f8038c212f396b0861d1a800de8854697b095d8545339c5', 'data/raw/taiwan_completion_20261108/coding_rules.json': '5fa3fb8dd3b6ff1b84bf8dcefb13bc927e247ee7bbbb37b07dbb2c6eae23bfdb', 'data/raw/taiwan_completion_20261108/g5_identity_search_00.txt': 'ec20911512b4ffba9c8231810710c19332e46fbc13f7a0b6b51b55ff411cf8fe', 'data/raw/taiwan_completion_20261108/g5_identity_search_01.txt': 'da4a3262fd48f0284f29452511b77269fc9a9f9e8b4472c4dcd4a845e1c27233', 'data/raw/taiwan_completion_20261108/mayor_final_followup.txt': 'e7420fab02926f280667c8c380a7be4481e17f42a85e20eefd450c3a86a06fcd', 'data/raw/taiwan_completion_20261108/mayor_followup_cao_zhong.txt': '7af38f43f28c281c3ee97cdcde790dbfdbc4459967b6fd5ebcf5706fd58b1f11', 'data/raw/taiwan_completion_20261108/mayor_query_plan.json': 'dad68414cf2ea6f20d769c70f299e57ff17ecadd284d002241f96f79d8e3c990', 'data/raw/taiwan_completion_20261108/mayor_search_000.txt': '624a984cf5df83fe03c12ff0df7c62304b4008a317f4f86c8975e4596257ac6e', 'data/raw/taiwan_completion_20261108/mayor_search_004.txt': '3dfa210bff741a04e28b73f1ec52deb207216349c6f5b9ff8339708e73c86b2c', 'data/raw/taiwan_completion_20261108/mayor_search_008.txt': '38acda2ebfd1a0230c0ed5e95f36787f177346e3e313b9da874e688fa77fb38a', 'data/raw/taiwan_completion_20261108/mayor_search_012.txt': '0e81dfc54991a5a1e10667e044ee5f268f696ea8d7114a316e3591c2884cbd6f', 'data/raw/taiwan_completion_20261108/mayor_search_016.txt': 'b9dc5065825dd5a58fa9dabae41d6f3aaceaa6030ef2b90dec1e753be6b253ef', 'data/raw/taiwan_completion_20261108/mayor_search_020.txt': '4a9fd05c13c3aa9648cb0bb6e1d6074a9be618d4c857a52a33b1077d9cbc6efd', 'data/raw/taiwan_completion_20261108/mayor_search_024.txt': '58f5f92c17c69edea5bd461a02947da2a55d03d6338339a76856ba5fcdeff8b3', 'data/raw/taiwan_completion_20261108/mayor_search_028.txt': 'c294f3ac9405337e8e1c1e9bd2ba9c2f6816a584a44f6bc655156bd071086060', 'data/raw/taiwan_completion_20261108/mayor_search_032.txt': '3ec27b1e8c22ba892cf7e3d521bd36533835cb075983f3efa7d1c01299e00b8d', 'data/raw/taiwan_completion_20261108/mayor_search_036.txt': 'f27115151b94de9286d51107e02d4c627ac32987a20cf57d94638e4a980f80c7', 'data/raw/taiwan_completion_20261108/mayor_search_040.txt': '30a97d531f6599433783a3343674a4143be1ebf4498820ba209bc65647bfcc01', 'data/raw/taiwan_completion_20261108/mayor_search_044.txt': '52f20d4e4a2ad10281dc3400ddbb1e398fea9ad2e8347cb7e03d0923759cd6db', 'data/raw/taiwan_completion_20261108/mayor_search_048.txt': 'efcb27e00da8a08cd48beddae35e178d71adc0cd0e52786c2ed8f984017e1086', 'data/raw/taiwan_completion_20261108/mayor_search_052.txt': 'f2c42753adc80bf8f4d06f0d9eb10e1ca7688093656fd3ccdc341406f3639e03', 'data/raw/taiwan_completion_20261108/mayor_search_056.txt': '24b626905ec910781ce3705a1813782a4c8c83439f71e61d4f1e28d1f2571e79', 'data/raw/taiwan_completion_20261108/mayor_search_060.txt': '8bdab2f8c2902188cbb3e172a7a7cc562ae25763f25b3fe9fa57688489f44eb5', 'data/raw/taiwan_completion_20261108/mayor_search_064.txt': '5723daf8d93c1ebc08af03692f9be65a8d4106ea63c6d9291aadfc8ca20b96d4', 'data/raw/taiwan_completion_20261108/mayor_search_068.txt': '6f38f8bb10e1f6a6b46944deb2d395e2034fc10fc088ddc601711fbc6fa2ae77', 'data/raw/taiwan_completion_20261108/mayor_search_072.txt': '6a5a37c8e4171731c9991f07617aa302a6883e817c0c2e732d2d18ac3465ddb6', 'data/raw/taiwan_completion_20261108/mayor_search_076.txt': 'ff9bf97bbdef548d57f79c993bee049909666939f5f13755b9b57ca2c8c44fcd', 'data/raw/taiwan_completion_20261108/mayor_search_080.txt': 'eb9b415c419680dff72509781c446e2faaa280d2565865bb65b96c8ee3d26a8a', 'data/raw/taiwan_completion_20261108/mayor_search_084.txt': 'd7607b4a839a63f8d2fbeddf503dae7f4f2a9e51b61afaedd961158bf964d121', 'data/raw/taiwan_completion_20261108/mayor_search_088.txt': '19e5bdf9ee0cc9678804ffa19c8ade11018957f15469fd006a446cc0a8d26c0a', 'data/raw/taiwan_completion_20261108/mayor_search_092.txt': 'a4985c2171edf06a9bc02ad405ba503087e081f61db760cdef8aca44c93ca66a', 'data/raw/taiwan_completion_20261108/mayor_search_096.txt': '934714da2177b029c964806c78eb2ec3db51deb09aa9c62ff6a5bf4b67f44071', 'data/raw/taiwan_completion_20261108/mayor_target_0.txt': 'a7606827c3046786b5d95aaca6a85635e090640696a1759e839fb8ed18abf1ef', 'data/raw/taiwan_completion_20261108/mayor_target_1.txt': '53ef546a724e194f8cae9de09d0166f2c4fd44714b03f8398c801828a3be4963', 'data/raw/taiwan_completion_20261108/qualification_snapshot_fulltext.txt': 'a764625ad73e55cb6524bebbf256967b23fb8c2e12e5513937209d03e9aec743', 'data/raw/taiwan_completion_20261108/regional_lead_search.txt': '4256d86a9a885e425e88ffc44d81988f208807a756c00403e7b8e187acac9d13', 'data/raw/taiwan_completion_20261108/regional_query_plan.json': '1bb43a4a452b96b4501bc14ec266bd4b404dc22b7704ab0d740e3a511dd3b9dc', 'data/raw/taiwan_completion_20261108/regional_search_000.txt': '04f893dffd5ae880d67e5540ab7eb8a2f7e1ab55907a13a6955e7ae21c5c1c06', 'data/raw/taiwan_completion_20261108/regional_search_004.txt': '3dd2e38afb35f5dfdcab45eece3fcebf9bcb6321734e5b990a77310ce52d0034', 'data/raw/taiwan_completion_20261108/regional_search_008.txt': '163b5a0880d9270ae25545bb61c53f678d6b2e3746e840bac76057830d5588d7', 'data/raw/taiwan_completion_20261108/regional_search_012.txt': 'bf8122b09c3ec7a2c76e9d0c0b65800a8f234cc0f79a51967e15f879fd6246d4', 'data/raw/taiwan_completion_20261108/regional_search_016.txt': '33003bd74a09ff4320b813674f819317e37fc6029eae9ff7a61d17f3cdb71ef3', 'data/raw/taiwan_completion_20261108/regional_search_020.txt': 'f2658f042c55460021aa1532cc49252356af700510fffce0fcb404ed393346dc', 'data/raw/taiwan_completion_20261108/regional_search_024.txt': 'f61c1f0e4fe98305442b774da1fe52e8779bfdd132d7bffb70acba978a95c678', 'data/raw/taiwan_completion_20261108/supplement_article_00.txt': 'f008e1e2d91d8d3f1642979ee2905a85bce35ebe39ad13383295b1f7112d1c5f', 'data/raw/taiwan_completion_20261108/supplement_article_01.txt': '3a8dc5a4ac49763c4fb5298fe8ed0857f55bcc688e78f6313d7454308bed8123', 'data/raw/taiwan_completion_20261108/supplement_article_02.txt': '43a22d643e4e1bd77cf71d4b6dacc14ae64b08e2f4970f1dc46c4cb789e660a6', 'data/raw/taiwan_completion_20261108/supplement_article_03.txt': '206e7dd1ef7afe615a78302c0fb419b70c96aae12303acfb2bbaffc8b952eb3e', 'data/raw/taiwan_completion_20261108/supplement_article_04.txt': '520035ae6f7218963869fae357f24bdd9ed85d50ca65dc8b20b21690285143a3', 'data/raw/taiwan_completion_20261108/supplement_article_05.txt': '0427ef72621a76888ae9dea1ceb13a380f25b077a9b0f6425afef5f0504149d2', 'data/raw/taiwan_completion_20261108/supplement_article_06.txt': '09bc31de573d46e14cbffab29d9cb3991585f55d5503259289ba3243f6ac486e', 'data/raw/taiwan_completion_20261108/supplement_article_07.txt': '09bc31de573d46e14cbffab29d9cb3991585f55d5503259289ba3243f6ac486e', 'data/raw/taiwan_completion_20261108/supplement_article_08.txt': '5dc5a9065437f0c83d94fcc26d9c8dd2b299767ecfe826692c186d8c8c12907d', 'data/raw/taiwan_completion_20261108/supplement_article_09.txt': '27ccb7dfeeb00c002e0f92f3fab9a490f05e4491b9bd7606508849d9ccf06d02', 'data/raw/taiwan_completion_20261108/supplement_article_10.txt': '119ecd2abcec99f6ab32e331e51287b2552817a61751108139543b006b60ecfb', 'data/raw/taiwan_completion_20261108/supplement_article_11.txt': '7767c3252d95f60426da1e8f303a587ad42ccfdde5f185cb8f2d7bbbd7967036', 'data/raw/taiwan_completion_20261108/supplement_article_12.txt': '904ba19c91d5f287ad06f45a9ba73a533e4504a73bdf83049510acf37a8af693', 'data/raw/taiwan_completion_20261108/supplement_article_14.txt': '595f4b3d6480dc226e7497da20e9c4abe9b9d443d0572bd654781e876b20d634', 'data/raw/taiwan_completion_20261108/supplement_article_15.txt': '48646a2236ad233fee372ea6df3686959e6dd5b484d5e9317703d3f3c0dba269', 'data/raw/taiwan_completion_20261108/supplement_article_16.txt': '0ec83114a1a764275ffa816213ab242edb53238e384aa03077fb925383660a85', 'data/raw/taiwan_completion_20261108/supplement_article_17.txt': '534896eb644095da96a5ebbb53c16c81e2f4f5c0bfaf29d96b5900d882813844', 'data/raw/taiwan_completion_20261108/supplement_article_18.txt': 'f7f50905fb854118693c97d74644ad592ffa826589049c29a8582faae3fa0dd6', 'data/raw/taiwan_completion_20261108/supplement_article_19.txt': 'c63cf7821c821708cb8c8182cc6c496e9cc8cade039f16c2418a995d161c8a73', 'data/raw/taiwan_completion_20261108/supplement_article_20.txt': '0631b7acca9a096dbeb376af58d3f9c822cb56e36a6baad0cbdc1b3cdc049e34', 'data/raw/taiwan_completion_20261108/supplement_article_21.txt': '6a1b0b233380b318241ec0937e5c132ff4fdd7d88ad4b9fd277fd284232d6028', 'data/raw/taiwan_completion_20261108/supplement_article_22.txt': 'f0c26ab3788b5e8d6ae533caa193185e05c9142a1ecdd0c5b20e4dfa55f13c78', 'data/raw/taiwan_completion_20261108/supplement_article_23.txt': '965e59073fb3e1368f5005c248483674dfe140559203b27a8a9fcc391b991622', 'data/raw/taiwan_completion_20261108/supplement_article_manifest.json': 'f0b16c52fdd3477a06e2aec59acb4211acd2ce402aad49343a120a92cc56e3ee'}
for rel,expected in (PINS|RAW_PINS).items():assert (ROOT/rel).is_file() and sha(ROOT/rel)==expected,rel
base=pd.read_csv(ROOT/'data/processed/clustering_all_20261108.csv',dtype=str,keep_default_na=False)
tk=pd.read_csv(ROOT/'data/processed/taipei_kaohsiung_yu_provisional_all_20261108.csv',dtype=str,keep_default_na=False)
assert base.shape==(1583,41) and tk.shape==(210,51)
assert base[KEY].is_unique and tk[KEY].is_unique and set(tk[KEY])<=set(base[KEY])
assert base.county_city.nunique()==22
before_original={f:int(base[f].eq('').sum()) for f in [G5,G6]}
d=base.set_index(KEY).copy()
for c in tk.columns:
    if c!=KEY and c not in d:d[c]=''
d.loc[tk[KEY],tk.columns.drop(KEY)]=tk.set_index(KEY)
d=d.reset_index()
assert len(d)==1583 and d[KEY].is_unique
pd.testing.assert_frame_equal(d.set_index(KEY).loc[tk[KEY],tk.columns.drop(KEY)],tk.set_index(KEY),check_names=False)
after_overlay={f:int(d[f].eq('').sum()) for f in [G5,G6]}
d['G5_inferred_archive_nonmatch_20261108']='0'
d['G5_completion_status_20261108']='prior_or_TK_evidence_retained'
d['G6_completion_status_20261108']='prior_or_TK_evidence_retained'
print('原全國缺值',before_original,'合併兩市後缺值',after_overlay)
display(d[[KEY,'candidate_name','county_city','office_type',G5,G6]].head(10))

原全國缺值 {'G5_first_time_any_elected_office': 640, 'G6_political_family': 1478} 合併兩市後缺值 {'G5_first_time_any_elected_office': 636, 'G6_political_family': 1296}


,candidate_key,candidate_name,county_city,office_type,G5_first_time_any_elected_office,G6_political_family
0,2026-C1-001,郭璽,臺北市,直轄市長,0,0
1,2026-C1-002,沈伯洋,臺北市,直轄市長,0,1
2,2026-C1-003,蕭文乾,臺北市,直轄市長,0,0
3,2026-C1-004,唐新民,臺北市,直轄市長,0,0
4,2026-C1-005,蔣萬安,臺北市,直轄市長,0,1
5,2026-C1-006,林志成,臺北市,直轄市長,0,0
6,2026-C1-007,蘇巧慧,新北市,直轄市長,0,1
7,2026-C1-008,李四川,新北市,直轄市長,1,
8,2026-C1-009,蘇輝湟,新北市,直轄市長,0,0
9,2026-C1-010,張善政,桃園市,直轄市長,0,


## G5 archive screening and identity adjudication

歷史資料只有「已存檔覆蓋」而非完整終生參選史。正規化空格／常見分隔符、臺／台，另掃描原住民中羅姓名的完整中文部分；這不等於已搜遍所有曾用名。145檔逐檔核SHA；CSV無header，第一列也納入。
無任何姓名／此別名命中者按學生選定的楊修訂規則給1，另標推論旗標；有同名但生日或學校／經歷未支持者保留未知。三項0須同時通過生日與閱讀完整人物頁的學校、歷次參選經歷連結。例示年份不是終生首次參選年份。

In [3]:
manifest=json.loads((ROOT/'data/raw/cec_all_offices_manifest.json').read_text())
assert len(manifest['members'])==146
# Two archive member labels refer to the same saved 2008 presidential CSV.
unique_members={}
for entry in manifest['members']:
    if entry['path'] in unique_members:assert unique_members[entry['path']]['sha256']==entry['sha256']
    else:unique_members[entry['path']]=entry
assert len(unique_members)==145
index={}
def norm(s):return re.sub(r'[\s．·・‧\-]','',str(s)).replace('台','臺')
def aliases(s):
    full=norm(s);chinese=norm(re.sub(r'[^\u3400-\u9fff]','',str(s)))
    return sorted({a for a in [full,chinese] if len(a)>=2})
def dob(s):
    s=s.strip()
    if re.fullmatch(r'\d{7}',s):
        return f'{int(s[:3])+1911:04d}-{s[3:5]}-{s[5:7]}'
    return ''
for entry in unique_members.values():
    assert sha(ROOT/entry['path'])==entry['sha256'],entry['path']
    with (ROOT/entry['path']).open(encoding='utf-8-sig',newline='') as f:
        for line,row in enumerate(csv.reader(f),1):
            assert len(row)>=10
            item={'path':entry['path'],'line':line,'name':row[6].strip(),'dob':dob(row[9]),'archive_member':entry['archive_member']}
            index.setdefault(norm(row[6]),[]).append(item)
manual=json.loads((ROOT/'data/raw/taiwan_completion_20261108/G5_identity_adjudications.json').read_text())
manual={x[KEY]:x for x in manual}
g5rows=[]
for i,row in d[d[G5].eq('')].iterrows():
    aa=aliases(row.candidate_name);hits={ (h['path'],h['line']):h for a in aa for h in index.get(a,[]) }
    matched_dob=[h for h in hits.values() if h['dob'] and h['dob']==row.birth_date]
    value='';status='pending_same_name_identity';source='';note='同名歷史命中，未經充分身分核實，不推為曾參選。'
    if not hits:
        value='1';status='inferred_no_match_available_archive';source='data/raw/cec_all_offices_manifest.json'
        note='已存145份CEC歷史資料無正規化姓名／中文別名命中；依楊修訂規則編1，非完整終生首次證明。'
        d.at[i,'G5_inferred_archive_nonmatch_20261108']='1'
    elif row[KEY] in manual:
        assert matched_dob and manual[row[KEY]]['value']=='0'
        doc=manual[row[KEY]]['source'];assert sha(ROOT/doc['path'])==doc['sha256']
        assert len(manual[row[KEY]]['identity_anchors'])>=3
        value='0';status='historical_identity_DOB_school_career_supported'
        chosen=matched_dob[0];source=f"{chosen['path']}:{chosen['line']}; {doc['url']}; {doc['path']}"
        note=manual[row[KEY]]['note'];d.at[i,'G5_prior_election_year_example']=re.search(r'/((?:19|20)\d{2})',chosen['archive_member']).group(1)
    d.at[i,'G5_completion_status_20261108']=status
    if value:
        d.at[i,G5]=value;d.at[i,'G5_label']='首次參選任何民選公職（可用歷史檔無命中推論）' if value=='1' else '非首次參選任何民選公職'
        d.at[i,'G5_evidence']=note;d.at[i,'G5_source']=source;d.at[i,'G5_audit_status']=status
    g5rows.append({KEY:row[KEY],'candidate_name':row.candidate_name,'county_city':row.county_city,'previous_value':'','value':value,'status':status,'aliases_screened':' | '.join(aa),'name_hits':len(hits),'exact_DOB_hits':len(matched_dob),'evidence_note':note,'source':source,'history_hits_json':json.dumps(list(hits.values()),ensure_ascii=False)})
g5audit=pd.DataFrame(g5rows)
assert len(g5audit)==636 and g5audit.status.value_counts().to_dict()=={'pending_same_name_identity':349,'inferred_no_match_available_archive':284,'historical_identity_DOB_school_career_supported':3}
display(g5audit.groupby(['status','value'],dropna=False).size().rename('N').reset_index())

,status,value,N
0,historical_identity_DOB_school_career_supported,0,3
1,inferred_no_match_available_archive,1,284
2,pending_same_name_identity,,349


## G6 reviewed search and qualifications

67位有逐人決策：56位指定搜尋未發現符合證據、10位完整文章支持親屬／公職、曹爾元1位待查。未搜尋的其他候選人仍空白。兩輪查詢按姓名、縣市及親屬／政治家族詞；來源能見度差異大，0的確定性不等於1，主分析不得推為真實家族盛行率。
資格資料沿用存檔的2026-10-16中選會API公告，並保存全文。9位非游智彬的資格不符者排除；游智彬保留既有暫納入旗標與正式紀錄。這是研究快照，不宣稱已複核往後的司法或選票名單。

In [4]:
adjudications=json.loads((ROOT/'data/raw/taiwan_completion_20261108/G6_adjudications.json').read_text())
g6rows=[]
for x in adjudications:
    ix=d.index[d[KEY].eq(x[KEY])];assert len(ix)==1;i=ix[0];assert d.at[i,G6]==''
    assert x['value'] in ['','0','1']
    if x['value']=='0':assert len(x['queries'])>=2 and x['status']=='searched_no_qualifying_evidence'
    if x['value']=='1':
        assert x['captures'] and x['status']=='positive_full_article_relationship_office'
        for s in x['captures']:assert sha(ROOT/s['path'])==s['sha256']
    d.at[i,'G6_completion_status_20261108']=x['status']
    if x['value']:
        d.at[i,G6]=x['value'];d.at[i,'G6_label']='有符合範圍公職親屬' if x['value']=='1' else '指定搜尋未發現符合範圍證據'
        d.at[i,'G6_evidence']=x['evidence_note'];d.at[i,'G6_source']='; '.join([s['url']+'; '+s['path'] for s in x['captures']]+['data/raw/taiwan_completion_20261108/G6_adjudications.json'])
        d.at[i,'G6_evidence_tier']='full_article_relationship_and_held_office' if x['value']=='1' else 'user_rule_targeted_search_nonfinding'
        d.at[i,'G6_search_status_20261108']=x['status']
    g6rows.append({KEY:x[KEY],'candidate_name':x['candidate_name'],'county_city':x['county_city'],'previous_value':'','value':x['value'],'status':x['status'],'evidence_note':x['evidence_note'],'query_count':len(x['queries']),'queries_json':json.dumps(x['queries'],ensure_ascii=False),'captures_json':json.dumps(x['captures'],ensure_ascii=False),'search_coverage':x['search_coverage']})
g6audit=pd.DataFrame(g6rows);assert len(g6audit)==67
qual_raw=json.loads((ROOT/'data/raw/taipei_kaohsiung_completion_20261108/cec_eligibility.json').read_text())
blocks=json.loads(qual_raw['data']['content'])['content']
qualification_text='\n'.join(BeautifulSoup(x.get('content',''),'html.parser').get_text(' ',strip=True) for x in blocks)
assert qualification_text==(ROOT/'data/raw/taiwan_completion_20261108/qualification_snapshot_fulltext.txt').read_text()
excluded_names=['林正峰','蕭湘瀚','鄭伯其','曹暐國','鍾任明','何博倫','周駿宥','石永城','張芳麗']
assert all(name in qualification_text for name in excluded_names+['游智彬'])
# Name is only a selector here; county+district+office are verified against the exact qualification text.
expected_scope={'林正峰':('新北市','第10選舉區'),'蕭湘瀚':('桃園市','第1選舉區'),'鄭伯其':('臺中市','第13選舉區'),'曹暐國':('臺南市','第7選舉區'),'鍾任明':('雲林縣','第4選舉區'),'何博倫':('嘉義縣','第6選舉區'),'周駿宥':('花蓮縣','第5選舉區'),'石永城':('金門縣','第1選舉區'),'張芳麗':('基隆市','第3選舉區')}
for name,(county,district) in expected_scope.items():
    q=d[d.candidate_name.eq(name)&d.county_city.eq(county)&d.electoral_district.eq(county+district)&d.office_type.str.contains('議員')];assert len(q)==1,(name,q.shape)
    assert county+'不符合規定者' in qualification_text and district+name in qualification_text
bad=d.candidate_name.isin(excluded_names)
assert bad.sum()==9
d['national_research_included_20261108']=(~bad).astype(int).astype(str)
d['national_exclusion_reason_20261108']=''
d.loc[bad,'national_exclusion_reason_20261108']='qualification_snapshot_not_approved_20261016'
# Fill inclusion metadata for non-TK rows, without changing the 210-row evidence.
for c,default in [('eligibility_provisional_flag_20261108','0'),('research_inclusion_status_20261108','retained_qualification_snapshot_20261016'),('eligibility_status_20261108','retained_qualification_snapshot_20261016')]:
    d.loc[d[c].eq(''),c]=default
for c in ['eligibility_status_20261108','research_inclusion_status_20261108']:d.loc[bad,c]='excluded_qualification_not_approved_20261016'
non_TK=~d[KEY].isin(tk[KEY])
d.loc[non_TK,'eligibility_source_20261108']='https://web.cec.gov.tw/central/article/65212; data/raw/taipei_kaohsiung_completion_20261108/cec_eligibility.json'
d.loc[bad,'official_eligibility_note_20261108']='2026-10-16中選會公告不符合登記規定；本輪沿用資格快照，不宣稱已更新司法狀態。'
assert d.eligibility_provisional_flag_20261108.eq('1').sum()==1
assert d.loc[d.eligibility_provisional_flag_20261108.eq('1'),'candidate_name'].tolist()==['游智彬']
# Existing TK values, including Yu's official and research notes, remain exactly intact.
pd.testing.assert_frame_equal(d.set_index(KEY).loc[tk[KEY],tk.columns.drop(KEY)],tk.set_index(KEY),check_names=False)

## Serialization, handoff and verification

In [5]:
roster=d.sort_values(KEY).reset_index(drop=True)
all_pool=roster[roster.national_research_included_20261108.eq('1')].reset_index(drop=True)
mayor=all_pool[all_pool.office_type.isin(['直轄市長','縣（市）長'])].reset_index(drop=True)
council=all_pool[all_pool.office_type.isin(['直轄市議員','縣（市）議員'])].reset_index(drop=True)
assert len(all_pool)==1574 and len(mayor)==81 and len(council)==1493
assert set(mayor[KEY]).isdisjoint(council[KEY]) and set(mayor[KEY])|set(council[KEY])==set(all_pool[KEY])
missing=~all_pool[[G5,G6]].isin(['0','1']).all(axis=1)
gap=all_pool.loc[missing,[KEY,'candidate_name','county_city','office_type',G5,G6,'G5_completion_status_20261108','G6_completion_status_20261108']].copy()
gap['missing_G5']=gap[G5].eq('').astype(int);gap['missing_G6']=gap[G6].eq('').astype(int)
coverage=[]
for pool,frame in [('all',all_pool),('mayor',mayor),('council',council)]:
    for county,q in frame.groupby('county_city',sort=True):
        valid=q[[G5,G6]].isin(['0','1']).all(axis=1)
        coverage.append({'pool':pool,'county_city':county,'research_roster_n':len(q),'complete_case_n':int(valid.sum()),'excluded_missing_n':int((~valid).sum()),'missing_G5_n':int(q[G5].eq('').sum()),'missing_G6_n':int(q[G6].eq('').sum()),'retention_percent':100*valid.mean(),'archive_inferred_G5_n':int(q.G5_inferred_archive_nonmatch_20261108.eq('1').sum())})
coverage=pd.DataFrame(coverage)
assert len(coverage)==66 and coverage.complete_case_n.gt(0).all()
outputs={'national_g5_g6_roster_20261108.csv':roster,'national_g5_g6_all_20261108.csv':all_pool,'national_g5_g6_mayor_20261108.csv':mayor,'national_g5_g6_council_20261108.csv':council,'national_g5_g6_g5_audit_20261108.csv':g5audit,'national_g5_g6_g6_audit_20261108.csv':g6audit,'national_g5_g6_gap_queue_20261108.csv':gap,'national_g5_g6_coverage_20261108.csv':coverage}
OUT=ROOT/'data/processed';saved=[]
for name,frame in outputs.items():
    p=OUT/name;frame.to_csv(p,index=False,encoding='utf-8-sig',lineterminator='\n')
    reread=pd.read_csv(p,dtype=str,keep_default_na=False)
    expected=frame.fillna('').astype(str).reset_index(drop=True)
    pd.testing.assert_frame_equal(reread,expected,check_dtype=False)
    saved.append({'path':str(p.relative_to(ROOT)),'rows':len(frame),'columns':list(frame),'sha256':sha(p),'verification':'serialized_reopened_all_cells_equal'})
(OUT/'national_g5_g6_preparation_manifest_20261108.json').write_text(json.dumps(saved,ensure_ascii=False,indent=2))
validation={'as_of':'2026-11-08','input_bindings':PINS,'raw_bindings':RAW_PINS,'archive_file_count_verified':145,'roster_n':len(roster),'qualification_excluded_n':9,'research_roster_n':len(all_pool),'pool_roster_n':{'all':len(all_pool),'mayor':len(mayor),'council':len(council)},'complete_case_n':{p:int(q[[G5,G6]].isin(['0','1']).all(axis=1).sum()) for p,q in [('all',all_pool),('mayor',mayor),('council',council)]},'missing_original':before_original,'missing_after_TK_overlay':after_overlay,'missing_after_this_pass_all_1583':{f:int(roster[f].eq('').sum()) for f in [G5,G6]},'missing_in_research_roster':{f:int(all_pool[f].eq('').sum()) for f in [G5,G6]},'new_G5_resolved':int(g5audit.value.ne('').sum()),'G5_archive_inferred_n':284,'new_G6_resolved':int(g6audit.value.ne('').sum()),'G6_pending_cases':g6audit[g6audit.value.eq('')][[KEY,'candidate_name','evidence_note']].to_dict('records'),'TK_210_all_original_cells_preserved':True,'Yu_provisionally_included':True,'all_22_counties_have_complete_case_each_pool':True,'all_nationwide_G6_searched':False,'ready_for_G5_G6_complete_case_clustering':True,'all_G1_to_G6_complete':False,'hash_and_schema_reopen_verified':True}
vp=OUT/'national_g5_g6_preparation_validation_20261108.json';vp.write_text(json.dumps(validation,ensure_ascii=False,indent=2));assert json.loads(vp.read_text())==validation
for rel,expected in (PINS|RAW_PINS).items():assert sha(ROOT/rel)==expected
print(json.dumps(validation['complete_case_n'],ensure_ascii=False));print('全1583筆補查後缺值',validation['missing_after_this_pass_all_1583']);print('PASS: 8 CSV files serialized and reopened; 2 verification JSON files read back.')
display(coverage)

{"all": 351, "mayor": 80, "council": 271}
全1583筆補查後缺值 {'G5_first_time_any_elected_office': 349, 'G6_political_family': 1230}
PASS: 8 CSV files serialized and reopened; 2 verification JSON files read back.


,pool,county_city,research_roster_n,complete_case_n,excluded_missing_n,missing_G5_n,missing_G6_n,retention_percent,archive_inferred_G5_n
0,all,南投縣,63,5,58,21,58,7.936508,12
1,all,嘉義市,42,7,35,9,35,16.666667,6
2,all,嘉義縣,54,5,49,17,48,9.259259,4
3,all,基隆市,62,7,55,10,55,11.290323,27
4,all,宜蘭縣,64,6,58,16,58,9.375000,14
...,...,...,...,...,...,...,...,...,...
61,council,苗栗縣,56,2,54,19,54,3.571429,9
62,council,連江縣,14,2,12,4,12,14.285714,0
63,council,金門縣,34,2,32,9,31,5.882353,13
64,council,雲林縣,72,1,71,24,71,1.388889,20
